# Phase 7 — Explainability Analysis (Grad-CAM)

Produces Grad-CAM-style heatmaps for Good / Bad / Failure predictions from both detectors to assess whether each model attends to the tumor region or to background/skull structures.

Assumes `model_yolo`, `model_frcnn`, `device`, `box_iou_np`, and the predict wrappers from notebook 04 are available in the session.

In [ ]:
!pip install grad-cam -q

In [ ]:
# pick one good / bad / failure sample per model, based on prediction IoU
import os, random
from PIL import Image

def classify_samples(predict_fn, images_dir, labels_dir, n_each=1):
    good, bad, failure = [], [], []
    img_files = list(os.listdir(images_dir))
    random.shuffle(img_files)
    for img_name in img_files:
        img_path = f"{images_dir}/{img_name}"
        lbl_path = f"{labels_dir}/{img_name.rsplit('.',1)[0]}.txt"
        img = Image.open(img_path)
        w, h = img.size
        gt_boxes = []
        if os.path.exists(lbl_path):
            for line in open(lbl_path).read().strip().splitlines():
                if not line: continue
                cls, xc, yc, bw, bh = map(float, line.split())
                x1, y1 = (xc-bw/2)*w, (yc-bh/2)*h
                x2, y2 = (xc+bw/2)*w, (yc+bh/2)*h
                gt_boxes.append([x1, y1, x2, y2])
        if not gt_boxes:
            continue
        pred_boxes = predict_fn(img_path)
        if not pred_boxes:
            if len(failure) < n_each:
                failure.append(img_path)
            continue
        best_iou = max(box_iou_np(pb, gt_boxes[0]) for pb in pred_boxes)
        if best_iou > 0.7 and len(good) < n_each:
            good.append(img_path)
        elif 0 < best_iou < 0.4 and len(bad) < n_each:
            bad.append(img_path)
        if len(good) >= n_each and len(bad) >= n_each and len(failure) >= n_each:
            break
    return {"good": good, "bad": bad, "failure": failure}

yolo_samples = classify_samples(yolo_predict_fn, "data/yolo/test/images", "data/yolo/test/labels")
frcnn_samples = classify_samples(frcnn_predict_fn, "data/yolo/test/images", "data/yolo/test/labels")
print("YOLO:", yolo_samples)
print("Faster R-CNN:", frcnn_samples)

In [ ]:
# Faster R-CNN essentially never produces a true zero-detection failure (very
# high recall); use its lowest-confidence prediction in the test set instead
def find_lowest_confidence(images_dir, labels_dir, max_search=100):
    import torchvision.transforms.functional as F
    img_files = list(os.listdir(images_dir))
    random.shuffle(img_files)
    lowest_score, lowest_img = 1.0, None
    for img_name in img_files[:max_search]:
        img_path = f"{images_dir}/{img_name}"
        img = Image.open(img_path).convert("RGB")
        img_tensor = F.to_tensor(img).to(device)
        with torch.no_grad():
            output = model_frcnn([img_tensor])[0]
        if len(output["scores"]) > 0:
            max_score = output["scores"].max().item()
            if max_score < lowest_score:
                lowest_score, lowest_img = max_score, img_path
    return lowest_img, lowest_score

low_conf_img, low_conf_score = find_lowest_confidence("data/yolo/test/images", "data/yolo/test/labels")
frcnn_samples["failure"] = [low_conf_img]
print(f"Lowest confidence: {low_conf_score:.3f} on {low_conf_img}")

In [ ]:
# EigenCAM for YOLO, implemented manually (first principal component of layer
# activations via SVD) since Ultralytics' multi-output detection head is
# incompatible with standard gradient-based CAM implementations
import cv2, numpy as np, torch
from pytorch_grad_cam.utils.image import show_cam_on_image

def yolo_eigencam_manual(img_path, model_yolo, device):
    target_layer = model_yolo.model.model[-2]
    activations = {}
    def hook(module, input, output):
        activations["value"] = output
    h = target_layer.register_forward_hook(hook)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (640, 640))
    rgb_img = img_resized.astype(np.float32) / 255.0
    input_tensor = torch.from_numpy(rgb_img).permute(2, 0, 1).unsqueeze(0).float().to(device)
    with torch.no_grad():
        model_yolo.model(input_tensor)
    h.remove()
    acts = activations["value"]
    if isinstance(acts, (list, tuple)):
        acts = acts[0]
    acts = acts[0].detach().cpu().numpy()
    reshaped = acts.reshape(acts.shape[0], -1).T
    reshaped = reshaped - reshaped.mean(axis=0)
    U, S, VT = np.linalg.svd(reshaped, full_matrices=False)
    cam = U[:, 0].reshape(acts.shape[1], acts.shape[2])
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (640, 640))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam_image = show_cam_on_image(rgb_img, cam, use_rgb=True)
    return img_resized, cam_image

In [ ]:
# custom gradient-based CAM for Faster R-CNN via forward/backward hooks on the
# last ResNet-50 backbone layer, backpropagating from the max detection score
import torchvision.transforms.functional as F

def frcnn_gradcam(img_path, model_frcnn, device):
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize((512, 512))
    rgb_img = np.array(img_resized).astype(np.float32) / 255.0
    img_tensor = F.to_tensor(img_resized).to(device)
    target_layer = model_frcnn.backbone.body.layer4[-1]
    activations, gradients = {}, {}
    def forward_hook(module, input, output):
        activations["value"] = output
    def backward_hook(module, grad_in, grad_out):
        gradients["value"] = grad_out[0]
    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)
    img_tensor.requires_grad_(True)
    output = model_frcnn([img_tensor])[0]
    if len(output["scores"]) == 0:
        h1.remove(); h2.remove()
        return img_resized, np.array(img_resized)
    score = output["scores"].max()
    model_frcnn.zero_grad()
    score.backward()
    h1.remove(); h2.remove()
    acts = activations["value"][0].detach().cpu().numpy()
    grads = gradients["value"][0].detach().cpu().numpy()
    weights = grads.mean(axis=(1, 2))
    cam = np.zeros(acts.shape[1:], dtype=np.float32)
    for i, w_ in enumerate(weights):
        cam += w_ * acts[i]
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (512, 512))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam_image = show_cam_on_image(rgb_img, cam, use_rgb=True)
    return img_resized, cam_image

In [ ]:
# build the final 3x2 grid: rows = Good/Bad/Failure, columns = YOLO/Faster R-CNN
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(10, 15))
row_labels = ["Good Prediction", "Bad Prediction", "Lowest-Confidence (Faster R-CNN) / Failure (YOLO)"]
case_keys = ["good", "bad", "failure"]

for row, (case, label) in enumerate(zip(case_keys, row_labels)):
    yolo_img_path = yolo_samples[case][0]
    frcnn_img_path = frcnn_samples[case][0]
    _, yolo_cam = yolo_eigencam_manual(yolo_img_path, model_yolo, device)
    _, frcnn_cam = frcnn_gradcam(frcnn_img_path, model_frcnn, device)
    axes[row, 0].imshow(yolo_cam); axes[row, 0].set_title(f"YOLOv8n — {label}", fontsize=10); axes[row, 0].axis("off")
    axes[row, 1].imshow(frcnn_cam); axes[row, 1].set_title(f"Faster R-CNN — {label}", fontsize=10); axes[row, 1].axis("off")

plt.tight_layout()
import os
os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/gradcam_comparison.png", dpi=150)
plt.show()

## Results

YOLOv8n produces a sharp, tumor-localized heatmap on good predictions and clearly diffuse activation on bad/failure predictions — an interpretable confidence signal. Faster R-CNN's heatmaps are more diffuse throughout, mixing tumor-region and image-border activation (a partial artifact of zero-padding in deep conv layers), and it essentially never produces a complete failure (lowest observed confidence: 0.904). Full discussion: final report, Section 9.